## 1. Environment & Configuration

Set up the discovery retrieval evaluation environment. Define the project paths, golden dataset location, existing Chroma vector database path, experiment configuration, and results directory.

This step only prepares the evaluation environment. It does not modify or re-index the existing vector database.

In [1]:

from pathlib import Path
from datetime import datetime
import json
import pandas as pd
import numpy as np

# Project directories
PROJECT_DIR = Path(r"G:\SEVA_Ai\SEVA_Ai")

GOLDEN_PATH = (
    PROJECT_DIR
    / "notebooks/golden_dataset/discovery/seva_discovery_golden_v1.jsonl"
)

DISCOVERY_DB_PATH = Path(r"G:\SEVA_Ai\SEVA_Ai\output\SEVA_CHILDREN\vectorstores\discovery")

RESULTS_DIR = (
    PROJECT_DIR
    / "notebooks/experiments/discovery_evaluation"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Experiment configuration
EXPERIMENT_ID = "DISC_EVAL_001"

TOP_K_VALUES = [5, 10, 20]

RETRIEVAL_CONFIG = {
    "search_type": "similarity",
    "top_k": 20,
}

print("Experiment:", EXPERIMENT_ID)
print("Golden dataset exists:", GOLDEN_PATH.exists())
print("Discovery DB exists:", DISCOVERY_DB_PATH.exists())
print("Results directory:", RESULTS_DIR)


Experiment: DISC_EVAL_001
Golden dataset exists: True
Discovery DB exists: True
Results directory: G:\SEVA_Ai\SEVA_Ai\notebooks\experiments\discovery_evaluation


## 2. Load Golden Dataset

Load the discovery golden dataset and validate its structure before running retrieval experiments.

Check the total number of questions, unique schemes, required fields, missing values, and duplicate questions. This ensures the evaluation dataset is ready for consistent retrieval testing.

In [6]:
# Load golden dataset
golden_df = pd.read_json(GOLDEN_PATH, lines=True)

# Validate required columns
required_columns = [
    "test_id",
    "scheme_id",
    "scheme_name",
    "question",
    "relevant_scheme_ids",
    "categories",
    "source",
    "label_status",
]

missing_columns = set(required_columns) - set(golden_df.columns)
assert not missing_columns, f"Missing columns: {missing_columns}"

# Dataset statistics
print("Total questions:", len(golden_df))
print("Unique schemes:", golden_df["scheme_id"].nunique())
print("Unique questions:", golden_df["question"].nunique())
print("Duplicate questions:", golden_df["question"].duplicated().sum())
print("Missing values:\n", golden_df[required_columns].isnull().sum())

# Inspect sample records
display(golden_df.head(5))

Total questions: 605
Unique schemes: 121
Unique questions: 605
Duplicate questions: 0
Missing values:
 test_id                0
scheme_id              0
scheme_name            0
question               0
relevant_scheme_ids    0
categories             0
source                 0
label_status           0
dtype: int64


,test_id,scheme_id,scheme_name,question,relevant_scheme_ids,categories,source,label_status
0,DISC-0001,3he_F5gB5uF80_MGg5Hz,Transport Loan Scheme,I need a loan to purchase a light motor vehicl...,[3he_F5gB5uF80_MGg5Hz],"[Transport & Infrastructure, Business & Entrep...",groq_generated,provisional_llm_generated
1,DISC-0002,3he_F5gB5uF80_MGg5Hz,Transport Loan Scheme,Looking for financing to buy a small van for g...,[3he_F5gB5uF80_MGg5Hz],"[Transport & Infrastructure, Business & Entrep...",groq_generated,provisional_llm_generated
2,DISC-0003,3he_F5gB5uF80_MGg5Hz,Transport Loan Scheme,Seeking funds to acquire an auto‑rickshaw to s...,[3he_F5gB5uF80_MGg5Hz],"[Transport & Infrastructure, Business & Entrep...",groq_generated,provisional_llm_generated
3,DISC-0004,3he_F5gB5uF80_MGg5Hz,Transport Loan Scheme,Want a term loan for a motorbike to expand my ...,[3he_F5gB5uF80_MGg5Hz],"[Transport & Infrastructure, Business & Entrep...",groq_generated,provisional_llm_generated
4,DISC-0005,3he_F5gB5uF80_MGg5Hz,Transport Loan Scheme,Need financial support to buy a light commerci...,[3he_F5gB5uF80_MGg5Hz],"[Transport & Infrastructure, Business & Entrep...",groq_generated,provisional_llm_generated


## 01. Verify Local Embedding Model

### Why are we implementing this?

We have downloaded all-MiniLM-L6-v2 from Hugging Face.

Before loading it, we need to verify that the model weights, tokenizer and configuration files exist in our local directory.

This ensures that we can load the model without downloading files again.

In [7]:
from pathlib import Path

MODEL_PATH = Path(
    r"G:\SEVA_Ai\SEVA_Ai\models\all-MiniLM-L6-v2"
)

required_files = [
    "config.json",
    "model.safetensors",
    "modules.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "vocab.txt",
    "1_Pooling/config.json"
]

missing_files = [
    file for file in required_files
    if not (MODEL_PATH / file).exists()
]

if not missing_files:
    print("Model files verified successfully!")
    print("Model location:", MODEL_PATH)
else:
    print("Missing files:", missing_files)

Model files verified successfully!
Model location: G:\SEVA_Ai\SEVA_Ai\models\all-MiniLM-L6-v2


In [4]:
import os
import time

os.environ["HF_HUB_OFFLINE"] = "1"

from langchain_huggingface import HuggingFaceEmbeddings

start_time = time.perf_counter()

embedding_model = HuggingFaceEmbeddings(
    model_name=str(MODEL_PATH),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

load_time = time.perf_counter() - start_time

test_embedding = embedding_model.embed_query(
    "What government schemes are available for farmers?"
)

print(f"Model loading time: {load_time:.2f} seconds")
print("Embedding dimensions:", len(test_embedding))
print("Model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loading time: 78.57 seconds
Embedding dimensions: 384
Model loaded successfully!


In [8]:
import json
import time
from pathlib import Path
from collections import defaultdict

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

VECTORSTORE_DIR = Path(r"G:\SEVA_Ai\SEVA_Ai\output\SEVA_CHILDREN\vectorstores")

discovery_db = Chroma(
    collection_name="seva_discovery_v1",
    embedding_function=embedding_model,
    persist_directory=str(VECTORSTORE_DIR / "discovery")
)

detail_db = Chroma(
    collection_name="seva_detail_v1",
    embedding_function=embedding_model,
    persist_directory=str(VECTORSTORE_DIR / "detail")
)

discovery_count = discovery_db._collection.count()
detail_count = detail_db._collection.count()

print("Discovery collection:", discovery_count)
print("Detail collection:", detail_count)
print("Total indexed chunks:", discovery_count + detail_count)

Discovery collection: 23438
Detail collection: 67945
Total indexed chunks: 91383


In [10]:
# Retrieval configuration
RETRIEVAL_K = 30

def retrieve_schemes(query: str, k: int = RETRIEVAL_K):
    start = time.perf_counter()

    results = discovery_db.similarity_search_with_score(
        query=query,
        k=k
    )

    latency = time.perf_counter() - start

    # Deduplicate chunks into ranked schemes
    ranked_schemes = []
    seen_scheme_ids = set()

    for rank, (doc, score) in enumerate(results, start=1):
        scheme_id = doc.metadata.get("scheme_id")

        if scheme_id and scheme_id not in seen_scheme_ids:
            seen_scheme_ids.add(scheme_id)

            ranked_schemes.append({
                "scheme_id": scheme_id,
                "rank": len(ranked_schemes) + 1,
                "chunk_rank": rank,
                "distance": float(score),
                "section": doc.metadata.get("child_type")
            })

    return {
        "query": query,
        "retrieved_chunks": len(results),
        "retrieved_schemes": ranked_schemes,
        "latency": latency
    }


# Test retrieval
test_query = golden_df.iloc[0]["question"]

test_result = retrieve_schemes(test_query)

print("Query:", test_query)
print("Retrieved chunks:", test_result["retrieved_chunks"])
print("Unique schemes:", len(test_result["retrieved_schemes"]))
print("Latency:", round(test_result["latency"], 3), "seconds")

for scheme in test_result["retrieved_schemes"][:10]:
    print(
        f"Rank: {scheme['rank']} | "
        f"Scheme ID: {scheme['scheme_id']} | "
        f"Chunk Rank: {scheme['chunk_rank']}"
    )

Query: I need a loan to purchase a light motor vehicle for my transport business.
Retrieved chunks: 30
Unique schemes: 19
Latency: 0.737 seconds
Rank: 1 | Scheme ID: ZaL-yJoBXu-iUKXqmUCT | Chunk Rank: 1
Rank: 2 | Scheme ID: GMZdU5kB0AGc0F0ifzmY | Chunk Rank: 2
Rank: 3 | Scheme ID: hqIa0pwBXu-iUKXq00o3 | Chunk Rank: 3
Rank: 4 | Scheme ID: 3he_F5gB5uF80_MGg5Hz | Chunk Rank: 6
Rank: 5 | Scheme ID: ksaOaZsB0AGc0F0ilf7I | Chunk Rank: 7
Rank: 6 | Scheme ID: dca3F5gB0AGc0F0iMjSR | Chunk Rank: 8
Rank: 7 | Scheme ID: oKLGz5oBXu-iUKXqxUAg | Chunk Rank: 10
Rank: 8 | Scheme ID: mcb3T5sB0AGc0F0iTP2L | Chunk Rank: 13
Rank: 9 | Scheme ID: 6aJOaZsBXu-iUKXqSUKJ | Chunk Rank: 14
Rank: 10 | Scheme ID: hca_2ZoB0AGc0F0iCPyl | Chunk Rank: 15


In [11]:
# Run retrieval for all golden questions
evaluation_results = []

for _, row in golden_df.iterrows():
    result = retrieve_schemes(row["question"])

    evaluation_results.append({
        "test_id": row["test_id"],
        "question": row["question"],
        "expected_scheme_id": row["scheme_id"],
        "relevant_scheme_ids": row["relevant_scheme_ids"],
        "categories": row["categories"],
        "retrieved_chunks": result["retrieved_chunks"],
        "retrieved_schemes": result["retrieved_schemes"],
        "ranked_scheme_ids": [
            item["scheme_id"]
            for item in result["retrieved_schemes"]
        ],
        "latency": result["latency"]
    })

# Convert results into DataFrame
retrieval_df = pd.DataFrame(evaluation_results)

# Save raw retrieval results
retrieval_output_path = RESULTS_DIR / "discovery_retrieval_results.jsonl"

retrieval_df.to_json(
    retrieval_output_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print("Total evaluated questions:", len(retrieval_df))
print("Average retrieval latency:", round(retrieval_df["latency"].mean(), 3), "seconds")
print("Average retrieved chunks:", round(retrieval_df["retrieved_chunks"].mean(), 2))
print("Average unique schemes:", round(
    retrieval_df["ranked_scheme_ids"].apply(len).mean(), 2
))
print("Saved:", retrieval_output_path)

KeyboardInterrupt: 

## 5. Configure Ragas Evaluation

Prepare Ragas for ID-based discovery retrieval evaluation.

Use IDBasedContextPrecision and IDBasedContextRecall to compare the ranked retrieved scheme IDs against the relevant scheme IDs in the golden dataset.

These metrics do not require an LLM evaluator because relevance is determined directly from scheme IDs.

In [13]:
import ragas

from ragas import SingleTurnSample
from ragas.metrics import (
    IDBasedContextPrecision,
    IDBasedContextRecall
)

print("Ragas version:", ragas.__version__)
print("ID-based precision: Available")
print("ID-based recall: Available")

Ragas version: 0.4.4
ID-based precision: Available
ID-based recall: Available


C:\Users\devendra\AppData\Local\Temp\ipykernel_11924\1960258666.py:4: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0 and has no direct equivalent in 'ragas.metrics.collections'.
  from ragas.metrics import (
C:\Users\devendra\AppData\Local\Temp\ipykernel_11924\1960258666.py:4: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0 and has no direct equivalent in 'ragas.metrics.collections'.
  from ragas.metrics import (


## 5. Configure Ragas ID-Based Metrics

Initialize Ragas ID-based context precision and context recall metrics.

In Ragas 0.4.4, these two metrics remain available through the legacy metrics API. The collections-based API does not expose equivalent ID-based classes in this release.

These metrics compare retrieved scheme IDs against the known relevant scheme IDs without requiring an LLM judge.

In [5]:
from ragas.metrics import (
    IDBasedContextPrecision,
    IDBasedContextRecall
)

id_precision = IDBasedContextPrecision()
id_recall = IDBasedContextRecall()

print("Precision:", type(id_precision).__name__)
print("Recall:", type(id_recall).__name__)

Precision: IDBasedContextPrecision
Recall: IDBasedContextRecall


C:\Users\devendra\AppData\Local\Temp\ipykernel_11760\1407634038.py:1: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0 and has no direct equivalent in 'ragas.metrics.collections'.
  from ragas.metrics import (
C:\Users\devendra\AppData\Local\Temp\ipykernel_11760\1407634038.py:1: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0 and has no direct equivalent in 'ragas.metrics.collections'.
  from ragas.metrics import (


## 6. Run Ragas ID-Based Retrieval Evaluation

Evaluate all 605 discovery queries using Ragas IDBasedContextPrecision and IDBasedContextRecall.

For each query:
- Compare ranked unique scheme IDs against the known relevant scheme IDs.
- Calculate ID-based context precision.
- Calculate ID-based context recall.
- Store individual query scores.

Aggregate the scores to measure overall discovery retrieval performance.

This evaluation uses direct ID matching and does not require an LLM judge.

In [13]:
from ragas import SingleTurnSample

result = Path(r"G:\SEVA_Ai\SEVA_Ai\notebooks\experiments\discovery_evaluation\discovery_retrieval_results.jsonl")
with open(result, "r") as f:
    retrieval_df = pd.read_json(f, lines=True)

ragas_results = []

for _, row in retrieval_df.iterrows():

    sample = SingleTurnSample(
        retrieved_context_ids=row["ranked_scheme_ids"],
        reference_context_ids=row["relevant_scheme_ids"]
    )

    precision_score = await id_precision.single_turn_ascore(sample)
    recall_score = await id_recall.single_turn_ascore(sample)

    ragas_results.append({
        "test_id": row["test_id"],
        "scheme_id": row["expected_scheme_id"],
        "precision": float(precision_score),
        "recall": float(recall_score)
    })

# Create results DataFrame
ragas_df = pd.DataFrame(ragas_results)

# Aggregate metrics
ragas_summary = pd.DataFrame([{
    "total_queries": len(ragas_df),
    "context_precision": ragas_df["precision"].mean(),
    "context_recall": ragas_df["recall"].mean()
}])

display(ragas_summary.style.format({
    "context_precision": "{:.4f}",
    "context_recall": "{:.4f}"
}))

# Save results
ragas_df.to_csv(
    RESULTS_DIR / "ragas_per_query_results.csv",
    index=False
)

ragas_summary.to_csv(
    RESULTS_DIR / "ragas_summary.csv",
    index=False
)

print("Ragas evaluation completed:", len(ragas_df))

,total_queries,context_precision,context_recall
0,605,0.0254,0.4860


Ragas evaluation completed: 605


## 7. Ranking-Based Retrieval Evaluation

Evaluate the ranking quality of the discovery retriever using MRR, NDCG@K, and Recall@K.

Calculate these metrics at K = 5, 10, and 20 using ranked unique scheme IDs.

The evaluation uses the known relevant scheme IDs from the provisional golden dataset. Results should be interpreted as initial retrieval benchmarks, not exhaustive relevance judgments.

In [14]:
import math

def evaluate_ranking(ranked_ids, relevant_ids, k):
    relevant_ids = set(relevant_ids)
    top_k = ranked_ids[:k]

    relevant_ranks = [
        rank
        for rank, scheme_id in enumerate(top_k, start=1)
        if scheme_id in relevant_ids
    ]

    recall = (
        len(relevant_ranks) / len(relevant_ids)
        if relevant_ids else 0
    )

    reciprocal_rank = (
        1 / relevant_ranks[0]
        if relevant_ranks else 0
    )

    dcg = sum(
        1 / math.log2(rank + 1)
        for rank in relevant_ranks
    )

    ideal_count = min(len(relevant_ids), k)

    idcg = sum(
        1 / math.log2(rank + 1)
        for rank in range(1, ideal_count + 1)
    )

    ndcg = dcg / idcg if idcg else 0

    return recall, reciprocal_rank, ndcg


ranking_results = []

for k in [5, 10, 20]:
    scores = [
        evaluate_ranking(
            row["ranked_scheme_ids"],
            row["relevant_scheme_ids"],
            k
        )
        for _, row in retrieval_df.iterrows()
    ]

    score_df = pd.DataFrame(
        scores,
        columns=["Recall", "RR", "NDCG"]
    )

    ranking_results.append({
        "K": k,
        "Recall@K": score_df["Recall"].mean(),
        "MRR@K": score_df["RR"].mean(),
        "NDCG@K": score_df["NDCG"].mean()
    })

ranking_summary = pd.DataFrame(ranking_results)

display(ranking_summary.style.format({
    "Recall@K": "{:.4f}",
    "MRR@K": "{:.4f}",
    "NDCG@K": "{:.4f}"
}))

ranking_summary.to_csv(
    RESULTS_DIR / "ranking_metrics_summary.csv",
    index=False
)

,K,Recall@K,MRR@K,NDCG@K
0,5,0.3091,0.1983,0.2259
1,10,0.3967,0.2102,0.2544
2,20,0.4694,0.2151,0.2726


## 8. Retrieval Depth Experiment

Investigate how retrieval depth affects discovery performance.

Evaluate the following Chroma retrieval depths:

- 10 chunks
- 20 chunks
- 30 chunks
- 50 chunks
- 100 chunks

For each retrieval depth:
1. Retrieve chunks using the existing similarity search.
2. Deduplicate results using scheme_id while preserving ranking.
3. Calculate Recall@5, Recall@10, Recall@20, MRR@K and NDCG@K.
4. Measure average retrieval latency and unique schemes retrieved.
5. Save per-query results and aggregate experiment metrics.

Keep the retrieval pipeline, embedding model, golden dataset and evaluation methodology unchanged.

The objective is to identify the retrieval depth that provides the best balance between retrieval quality and latency.

In [20]:
# Experiment configuration
RETRIEVAL_DEPTHS = [30]
EVALUATION_K = [5, 10, 20]

depth_results = []
depth_summary = []

for depth in RETRIEVAL_DEPTHS:

    print(f"\nRunning retrieval depth: {depth}")

    for _, row in golden_df.iterrows():

        result = retrieve_schemes(
            query=row["question"],
            k=depth
        )

        ranked_ids = [
            item["scheme_id"]
            for item in result["retrieved_schemes"]
        ]

        record = {
            "test_id": row["test_id"],
            "retrieval_depth": depth,
            "relevant_scheme_ids": row["relevant_scheme_ids"],
            "ranked_scheme_ids": ranked_ids,
            "retrieved_chunks": result["retrieved_chunks"],
            "unique_schemes": len(ranked_ids),
            "latency": result["latency"]
        }

        depth_results.append(record)

    # Evaluate this retrieval depth
    current_results = [
        r for r in depth_results
        if r["retrieval_depth"] == depth
    ]

    summary = {
        "retrieval_depth": depth,
        "avg_chunks": np.mean([
            r["retrieved_chunks"] for r in current_results
        ]),
        "avg_unique_schemes": np.mean([
            r["unique_schemes"] for r in current_results
        ]),
        "avg_latency": np.mean([
            r["latency"] for r in current_results
        ])
    }

    for k in EVALUATION_K:

        scores = [
            evaluate_ranking(
                r["ranked_scheme_ids"],
                r["relevant_scheme_ids"],
                k
            )
            for r in current_results
        ]

        score_df = pd.DataFrame(
            scores,
            columns=["Recall", "RR", "NDCG"]
        )

        summary[f"Recall@{k}"] = score_df["Recall"].mean()
        summary[f"MRR@{k}"] = score_df["RR"].mean()
        summary[f"NDCG@{k}"] = score_df["NDCG"].mean()

    depth_summary.append(summary)

    print(
        f"Completed {depth}: "
        f"Average latency = {summary['avg_latency']:.3f}s"
    )


# Create experiment DataFrames
depth_results_df = pd.DataFrame(depth_results)
depth_summary_df = pd.DataFrame(depth_summary)

# Save experiment outputs
depth_results_df.to_json(
    RESULTS_DIR / "retrieval_depth_per_query.jsonl",
    orient="records",
    lines=True
)

depth_summary_df.to_csv(
    RESULTS_DIR / "retrieval_depth_summary.csv",
    index=False
)

# Display results
display(
    depth_summary_df.style.format({
        col: "{:.4f}"
        for col in depth_summary_df.select_dtypes(
            include="number"
        ).columns
        if col != "retrieval_depth"
    })
)

print("\nRetrieval depth experiment completed.")


Running retrieval depth: 30
Completed 30: Average latency = 0.185s


,retrieval_depth,avg_chunks,avg_unique_schemes,avg_latency,Recall@5,MRR@5,NDCG@5,Recall@10,MRR@10,NDCG@10,Recall@20,MRR@20,NDCG@20
0,30,30.0000,21.3174,0.1846,0.3091,0.1983,0.2259,0.3967,0.2102,0.2544,0.4694,0.2151,0.2726



Retrieval depth experiment completed.


Recommended retrieval depth
Baseline

30 chunks
- Average unique schemes: 21.32
- Recall@20: 46.94%
- No measured recall improvement at 50 or 100 chunks.

## 9. Discovery Retrieval Error Analysis

Identify queries where the known relevant scheme was not retrieved within the top 20 unique schemes.

Inspect:
- Citizen question
- Expected scheme ID and name
- Relevant scheme rank, if retrieved outside the top 20
- Retrieved scheme IDs
- Scheme categories

Export missed queries for manual inspection.

The objective is to identify retrieval failure patterns before implementing hybrid search or reranking.

In [21]:
# Select results from retrieval depth 30
baseline_df = depth_results_df[
    depth_results_df["retrieval_depth"] == 30
].copy()

# Find the rank of the expected relevant scheme
def find_relevant_rank(ranked_ids, relevant_ids):
    relevant_ids = set(relevant_ids)

    for rank, scheme_id in enumerate(ranked_ids, start=1):
        if scheme_id in relevant_ids:
            return rank

    return None


baseline_df["relevant_rank"] = baseline_df.apply(
    lambda row: find_relevant_rank(
        row["ranked_scheme_ids"],
        row["relevant_scheme_ids"]
    ),
    axis=1
)

# Join golden dataset metadata
missed_df = baseline_df.merge(
    golden_df[
        ["test_id", "scheme_id", "scheme_name", "question", "categories"]
    ],
    on="test_id",
    how="left"
)

# Identify queries where relevant scheme is not in top 20
missed_df = missed_df[
    missed_df["relevant_rank"].isna()
    | (missed_df["relevant_rank"] > 20)
].copy()

# Export missed queries
missed_output = RESULTS_DIR / "discovery_missed_queries.csv"

missed_df[[
    "test_id",
    "question",
    "scheme_id",
    "scheme_name",
    "categories",
    "relevant_rank",
    "ranked_scheme_ids"
]].to_csv(missed_output, index=False)

print("Total queries:", len(baseline_df))
print("Missed queries:", len(missed_df))
print(
    "Miss rate:",
    round(len(missed_df) / len(baseline_df) * 100, 2),
    "%"
)
print("Saved:", missed_output)

display(
    missed_df[
        ["test_id", "question", "scheme_name", "categories", "relevant_rank"]
    ].head(15)
)

Total queries: 605
Missed queries: 321
Miss rate: 53.06 %
Saved: G:\SEVA_Ai\SEVA_Ai\notebooks\experiments\discovery_evaluation\discovery_missed_queries.csv


,test_id,question,scheme_name,categories,relevant_rank
1,DISC-0002,Looking for financing to buy a small van for g...,Transport Loan Scheme,"[Transport & Infrastructure, Business & Entrep...",NaN
2,DISC-0003,Seeking funds to acquire an auto‑rickshaw to s...,Transport Loan Scheme,"[Transport & Infrastructure, Business & Entrep...",NaN
4,DISC-0005,Need financial support to buy a light commerci...,Transport Loan Scheme,"[Transport & Infrastructure, Business & Entrep...",NaN
5,DISC-0006,Requesting financial support to host a symposi...,Scheme for Celebration of Birth/death Annivers...,[Business & Entrepreneurship],NaN
8,DISC-0009,Looking for financial aid to organize a debate...,Scheme for Celebration of Birth/death Annivers...,[Business & Entrepreneurship],NaN
9,DISC-0010,Requesting grant to cover expenses for a confe...,Scheme for Celebration of Birth/death Annivers...,[Business & Entrepreneurship],NaN
10,DISC-0011,I need financial help for my cancer treatment ...,Delhi Arogya Kosh,[Health & Wellness],NaN
11,DISC-0012,Can I receive assistance for my heart surgery ...,Delhi Arogya Kosh,[Health & Wellness],NaN
13,DISC-0014,Looking for financial aid to cover dialysis ex...,Delhi Arogya Kosh,[Health & Wellness],NaN
14,DISC-0015,Need assistance for mammography test costs at ...,Delhi Arogya Kosh,[Health & Wellness],NaN


## 10. Investigate Retrieval Failures

Analyze queries where the expected scheme was not retrieved within the top 30 chunks.

Check whether the expected scheme:
- Is completely absent from the retrieved results.
- Appears in the retrieved results but is ranked below the top 20.
- Has multiple queries failing for the same scheme.

Identify repeated failure patterns to help determine whether the issue is related to query wording, scheme content, chunking, or semantic retrieval.

## 11. Build BM25 Sparse Retrieval Index

Build a BM25 keyword retrieval index using all existing documents from the Discovery Chroma collection.

Use the existing 23,438 indexed chunks without modifying the Chroma database.

BM25 will provide lexical retrieval alongside the existing dense vector retrieval.

Preserve scheme_id and other document metadata to support scheme-level deduplication and ranking fusion.

In [15]:
import re
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever

# Fetch all discovery documents from Chroma
discovery_data = discovery_db.get(
    include=["documents", "metadatas"]
)

# Convert Chroma records into LangChain documents
discovery_documents = [
    Document(
        page_content=content,
        metadata=metadata or {}
    )
    for content, metadata in zip(
        discovery_data["documents"],
        discovery_data["metadatas"]
    )
    if content and metadata and metadata.get("scheme_id")
]

# BM25 tokenizer
def bm25_tokenize(text):
    return re.findall(r"\w+", text.lower())

# Build BM25 index
bm25_retriever = BM25Retriever.from_documents(
    discovery_documents,
    k=30,
    preprocess_func=bm25_tokenize
)

print("Chroma documents:", len(discovery_data["documents"]))
print("BM25 indexed documents:", len(discovery_documents))
print("Unique schemes:", len({
    doc.metadata["scheme_id"]
    for doc in discovery_documents
}))
print("BM25 retrieval depth:", bm25_retriever.k)

Chroma documents: 23438
BM25 indexed documents: 23438
Unique schemes: 4428
BM25 retrieval depth: 30


## 12. BM25 Retrieval Evaluation Using Ragas

Evaluate BM25 sparse retrieval using Ragas IDBasedContextPrecision and IDBasedContextRecall.

Calculate both metrics at K = 5, 10, 20 and 30.

For each query:
1. Take the top K unique retrieved scheme IDs.
2. Compare them against the reference relevant scheme IDs.
3. Calculate Ragas context precision and context recall.
4. Store individual query scores.
5. Calculate aggregate scores for each K.

This provides a consistent evaluation methodology for comparing Dense, BM25 and Hybrid retrieval.

In [11]:
from ragas import SingleTurnSample

RAGAS_K_VALUES = [5, 10, 20, 30]

bm25_ragas_results = []

for _, row in bm25_df.iterrows():

    for k in RAGAS_K_VALUES:

        retrieved_ids = row["ranked_scheme_ids"][:k]

        sample = SingleTurnSample(
            retrieved_context_ids=retrieved_ids,
            reference_context_ids=row["relevant_scheme_ids"]
        )

        precision = await id_precision.single_turn_ascore(sample)
        recall = await id_recall.single_turn_ascore(sample)

        bm25_ragas_results.append({
            "test_id": row["test_id"],
            "K": k,
            "context_precision": float(precision),
            "context_recall": float(recall)
        })


# Per-query results
bm25_ragas_df = pd.DataFrame(bm25_ragas_results)

# Aggregate Ragas metrics
bm25_ragas_summary = (
    bm25_ragas_df
    .groupby("K")
    .agg(
        context_precision=("context_precision", "mean"),
        context_recall=("context_recall", "mean")
    )
    .reset_index()
)

display(
    bm25_ragas_summary.style.format({
        "context_precision": "{:.4f}",
        "context_recall": "{:.4f}"
    })
)

# Save results
bm25_ragas_df.to_csv(
    RESULTS_DIR / "bm25_ragas_per_query.csv",
    index=False
)

bm25_ragas_summary.to_csv(
    RESULTS_DIR / "bm25_ragas_summary.csv",
    index=False
)

print("Evaluated queries:", bm25_df["test_id"].nunique())
print("Ragas metric evaluations:", len(bm25_ragas_df))

,K,context_precision,context_recall
0,5,0.0833,0.4165
1,10,0.0491,0.4909
2,20,0.0292,0.5603
3,30,0.0260,0.5802


Evaluated queries: 605
Ragas metric evaluations: 2420


## 13. Hybrid Retrieval Using Reciprocal Rank Fusion

Combine Dense and BM25 retrieval rankings using Reciprocal Rank Fusion (RRF).

For each citizen question:

1. Retrieve the ranked unique scheme IDs from Dense retrieval.
2. Retrieve the ranked unique scheme IDs from BM25.
3. Calculate RRF scores using both rankings.
4. Combine the scores for matching scheme IDs.
5. Sort schemes by their combined RRF scores.

Use an RRF constant of 60.

Evaluate the resulting hybrid rankings using Ragas ID-based context precision and context recall.

The objective is to determine whether combining semantic and lexical retrieval improves discovery performance compared with either method independently.

In [22]:
RRF_K = 60

def reciprocal_rank_fusion(
    dense_ids,
    bm25_ids,
    rrf_k=RRF_K
):
    scores = {}
    first_seen = {}

    rankings = [dense_ids, bm25_ids]

    for ranking in rankings:
        for rank, scheme_id in enumerate(ranking, start=1):

            if scheme_id not in scores:
                scores[scheme_id] = 0.0
                first_seen[scheme_id] = len(first_seen)

            scores[scheme_id] += 1 / (rrf_k + rank)

    ranked_schemes = sorted(
        scores,
        key=lambda sid: (-scores[sid], first_seen[sid])
    )

    return [
        {
            "scheme_id": scheme_id,
            "rrf_score": scores[scheme_id],
            "rank": rank
        }
        for rank, scheme_id in enumerate(ranked_schemes, start=1)
    ]


# Get dense retrieval results at depth 30
dense_df = depth_results_df[
    depth_results_df["retrieval_depth"] == 30
].copy()

# Combine Dense and BM25 results
hybrid_results = []

for _, dense_row in dense_df.iterrows():

    bm25_row = bm25_df[
        bm25_df["test_id"] == dense_row["test_id"]
    ].iloc[0]

    fused = reciprocal_rank_fusion(
        dense_ids=dense_row["ranked_scheme_ids"],
        bm25_ids=bm25_row["ranked_scheme_ids"]
    )

    hybrid_results.append({
        "test_id": dense_row["test_id"],
        "relevant_scheme_ids": dense_row["relevant_scheme_ids"],
        "ranked_scheme_ids": [
            item["scheme_id"] for item in fused
        ],
        "rrf_results": fused
    })


hybrid_df = pd.DataFrame(hybrid_results)

# Save hybrid retrieval results
hybrid_df.to_json(
    RESULTS_DIR / "hybrid_rrf_results.jsonl",
    orient="records",
    lines=True
)

print("Hybrid queries:", len(hybrid_df))
print("Average unique schemes:", round(
    hybrid_df["ranked_scheme_ids"].apply(len).mean(), 2
))

print("\nSample fused ranking:")
display(pd.DataFrame(hybrid_df.iloc[0]["rrf_results"]).head(10))

Hybrid queries: 605
Average unique schemes: 41.02

Sample fused ranking:


,scheme_id,rrf_score,rank
0,3he_F5gB5uF80_MGg5Hz,0.032018,1
1,GMZdU5kB0AGc0F0ifzmY,0.031754,2
2,hqIa0pwBXu-iUKXq00o3,0.031746,3
3,hca_2ZoB0AGc0F0iCPyl,0.029437,4
4,ZaL-yJoBXu-iUKXqmUCT,0.016393,5
5,GMZscpoB0AGc0F0iyj3X,0.016129,6
6,ksaOaZsB0AGc0F0ilf7I,0.015385,7
7,YReeK5oB5uF80_MGtZWO,0.015385,8
8,dca3F5gB0AGc0F0iMjSR,0.015152,9
9,oKLGz5oBXu-iUKXqxUAg,0.014925,10


## 14. Hybrid Retrieval Evaluation Using Ragas

Evaluate the hybrid RRF retrieval results using Ragas IDBasedContextPrecision and IDBasedContextRecall.

Calculate both metrics at K = 5, 10, 20 and 30.

For each query:
1. Select the top K unique scheme IDs from the fused ranking.
2. Compare them against the reference relevant scheme IDs.
3. Calculate context precision and context recall.
4. Store individual query scores.
5. Calculate aggregate metrics for each K.

Use the same golden dataset and evaluation methodology as Dense and BM25 retrieval to ensure a fair comparison.

In [23]:
hybrid_ragas_results = []

for _, row in hybrid_df.iterrows():

    for k in RAGAS_K_VALUES:

        sample = SingleTurnSample(
            retrieved_context_ids=row["ranked_scheme_ids"][:k],
            reference_context_ids=row["relevant_scheme_ids"]
        )

        precision = await id_precision.single_turn_ascore(sample)
        recall = await id_recall.single_turn_ascore(sample)

        hybrid_ragas_results.append({
            "test_id": row["test_id"],
            "K": k,
            "context_precision": float(precision),
            "context_recall": float(recall)
        })

# Per-query results
hybrid_ragas_df = pd.DataFrame(hybrid_ragas_results)

# Aggregate results
hybrid_ragas_summary = (
    hybrid_ragas_df
    .groupby("K")
    .agg(
        context_precision=("context_precision", "mean"),
        context_recall=("context_recall", "mean")
    )
    .reset_index()
)

display(
    hybrid_ragas_summary.style.format({
        "context_precision": "{:.4f}",
        "context_recall": "{:.4f}"
    })
)

# Save results
hybrid_ragas_df.to_csv(
    RESULTS_DIR / "hybrid_ragas_per_query.csv",
    index=False
)

hybrid_ragas_summary.to_csv(
    RESULTS_DIR / "hybrid_ragas_summary.csv",
    index=False
)

print("Evaluated queries:", hybrid_df["test_id"].nunique())
print("Total Ragas evaluations:", len(hybrid_ragas_df))
print("Results saved successfully.")

,K,context_precision,context_recall
0,5,0.0826,0.4132
1,10,0.0512,0.5124
2,20,0.0309,0.6149
3,30,0.0227,0.6562


Evaluated queries: 605
Total Ragas evaluations: 2420
Results saved successfully.


## 15. Compare Dense, BM25 and Hybrid Retrieval

Compare all three retrieval approaches using Ragas IDBasedContextPrecision and IDBasedContextRecall.

Evaluate Dense retrieval at K = 5, 10, 20 and 30 to match the existing BM25 and Hybrid evaluations.

Combine the aggregate results into a single comparison table.

Identify which retrieval strategy provides the best context recall and precision at each cutoff.

In [24]:
# Evaluate Dense retrieval using the same Ragas metrics
dense_ragas_results = []

for _, row in retrieval_df.iterrows():

    for k in RAGAS_K_VALUES:

        sample = SingleTurnSample(
            retrieved_context_ids=row["ranked_scheme_ids"][:k],
            reference_context_ids=row["relevant_scheme_ids"]
        )

        precision = await id_precision.single_turn_ascore(sample)
        recall = await id_recall.single_turn_ascore(sample)

        dense_ragas_results.append({
            "test_id": row["test_id"],
            "K": k,
            "context_precision": float(precision),
            "context_recall": float(recall)
        })

dense_ragas_df = pd.DataFrame(dense_ragas_results)

dense_ragas_summary = (
    dense_ragas_df
    .groupby("K")
    .agg(
        context_precision=("context_precision", "mean"),
        context_recall=("context_recall", "mean")
    )
    .reset_index()
)

# Standardize all three summaries
dense_comparison = dense_ragas_summary.copy()
dense_comparison["retrieval"] = "Dense"

bm25_comparison = bm25_ragas_summary.copy()
bm25_comparison["retrieval"] = "BM25"

hybrid_comparison = hybrid_ragas_summary.copy()
hybrid_comparison["retrieval"] = "Hybrid RRF"

# Combine
final_comparison = pd.concat(
    [
        dense_comparison,
        bm25_comparison,
        hybrid_comparison
    ],
    ignore_index=True
)

final_comparison = final_comparison[
    ["retrieval", "K", "context_precision", "context_recall"]
].sort_values(["K", "retrieval"])

display(
    final_comparison.style.format({
        "context_precision": "{:.4f}",
        "context_recall": "{:.4f}"
    })
)

# Save comparison
final_comparison.to_csv(
    RESULTS_DIR / "dense_bm25_hybrid_comparison.csv",
    index=False
)

dense_ragas_df.to_csv(
    RESULTS_DIR / "dense_ragas_per_query.csv",
    index=False
)

print("Final comparison completed.")

,retrieval,K,context_precision,context_recall
4,BM25,5,0.0833,0.4165
0,Dense,5,0.0618,0.3091
8,Hybrid RRF,5,0.0826,0.4132
5,BM25,10,0.0491,0.4909
1,Dense,10,0.0397,0.3967
9,Hybrid RRF,10,0.0512,0.5124
6,BM25,20,0.0292,0.5603
2,Dense,20,0.0263,0.4694
10,Hybrid RRF,20,0.0309,0.6149
7,BM25,30,0.0260,0.5802


Final comparison completed.


## 16. Cross-Encoder Reranking

Introduce a Cross-Encoder reranker after Dense and BM25 retrieval results have been combined using RRF.

Use the existing cross-encoder model:

cross-encoder/ms-marco-MiniLM-L-6-v2

For each citizen query:
1. Take the candidate schemes from RRF.
2. Obtain the corresponding scheme document text.
3. Score each query-document pair using the Cross-Encoder.
4. Sort candidates by relevance score.
5. Preserve scheme IDs and evaluate the reranked results using Ragas.

The objective is to improve the ranking of relevant government schemes while preserving the candidate coverage achieved by hybrid retrieval.

In [25]:
from collections import defaultdict
from sentence_transformers import CrossEncoder

# Group discovery chunks by scheme
scheme_chunks = defaultdict(list)

for doc in discovery_documents:
    scheme_id = doc.metadata.get("scheme_id")

    if scheme_id:
        scheme_chunks[scheme_id].append(doc.page_content)

# Create representative text for each scheme
scheme_profiles = {}

for scheme_id, chunks in scheme_chunks.items():

    unique_chunks = list(dict.fromkeys(
        text.strip() for text in chunks if text.strip()
    ))

    representative_text = "\n".join(unique_chunks)

    scheme_profiles[scheme_id] = representative_text[:3500]


# Load Cross-Encoder
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

cross_encoder = CrossEncoder(
    RERANKER_MODEL,
    max_length=512
)

print("Total scheme profiles:", len(scheme_profiles))
print("Cross-Encoder:", RERANKER_MODEL)
print("Model loaded successfully.")

# Inspect one profile
sample_scheme_id = next(iter(scheme_profiles))

print("\nSample scheme ID:", sample_scheme_id)
print("Representative text:")
print(scheme_profiles[sample_scheme_id][:500])

Could not reach the Hub (Cannot reach https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L-6-v2: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'cross-encoder/ms-marco-MiniLM-L-6-v2'.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Total scheme profiles: 4428
Cross-Encoder: cross-encoder/ms-marco-MiniLM-L-6-v2
Model loaded successfully.

Sample scheme ID: LBezF5gB5uF80_MGGY81
Representative text:
Scheme: Stand-Up India
Section: overview
A scheme by Ministry of Finance for financing SC/ST and Women Entrepreneurs by facilitating bank loans for setting up a greenfield project Enterprise in manufacturing, services, trading sector and activities allied to agriculture.
Scheme: Stand-Up India
Section: benefits
1. Facilitation of composite loan (inclusive of term loan and working capital) between ₹10 Lakhs and ₹100 Lakhs. Rupay debit card to be issued for convenience of the borrower.
1. The web 


### Cross-Encoder Reranking

Rerank Hybrid RRF candidate schemes using a Cross-Encoder.

The Cross-Encoder evaluates the relevance between the user's query and each scheme's representative text. It produces a relevance score, allowing us to reorder candidate schemes based on query-specific relevance.

In [26]:
def rerank_schemes(query, candidate_scheme_ids, top_k=None):
    pairs = []
    valid_scheme_ids = []

    for scheme_id in candidate_scheme_ids:
        profile = scheme_profiles.get(scheme_id)

        if profile:
            pairs.append((query, profile))
            valid_scheme_ids.append(scheme_id)

    if not pairs:
        return []

    scores = cross_encoder.predict(
        pairs,
        batch_size=16,
        show_progress_bar=False
    )

    reranked = [
        {
            "scheme_id": scheme_id,
            "rerank_score": float(score)
        }
        for scheme_id, score in zip(valid_scheme_ids, scores)
    ]

    reranked.sort(
        key=lambda item: item["rerank_score"],
        reverse=True
    )

    if top_k:
        reranked = reranked[:top_k]

    return [
        {
            **item,
            "rank": rank
        }
        for rank, item in enumerate(reranked, start=1)
    ]

In [28]:
print("Columns:")
print(hybrid_df.columns.tolist())

print("\nFirst row:")
display(hybrid_df.head(1))

Columns:
['test_id', 'relevant_scheme_ids', 'ranked_scheme_ids', 'rrf_results']

First row:


,test_id,relevant_scheme_ids,ranked_scheme_ids,rrf_results
0,DISC-0001,[3he_F5gB5uF80_MGg5Hz],"[3he_F5gB5uF80_MGg5Hz, GMZdU5kB0AGc0F0ifzmY, h...","[{'scheme_id': '3he_F5gB5uF80_MGg5Hz', 'rrf_sc..."


### Connect Golden Questions with Hybrid RRF Results

Use test_id to retrieve the original question from golden_df and pair it with the candidate scheme IDs from hybrid_df.

In [29]:
# Create a lookup from test_id to original question
question_lookup = golden_df.set_index("test_id")["question"].to_dict()

# Select one Hybrid RRF result
sample = hybrid_df.iloc[0]

sample_query = question_lookup[sample["test_id"]]
candidate_ids = sample["ranked_scheme_ids"]

# Apply Cross-Encoder reranking
reranked_results = rerank_schemes(
    query=sample_query,
    candidate_scheme_ids=candidate_ids
)

print("Query:", sample_query)
print("\nTop 5 reranked schemes:")

for item in reranked_results[:5]:
    print(item)

Query: I need a loan to purchase a light motor vehicle for my transport business.

Top 5 reranked schemes:
{'scheme_id': '3he_F5gB5uF80_MGg5Hz', 'rerank_score': 5.329038619995117, 'rank': 1}
{'scheme_id': 'hqIa0pwBXu-iUKXq00o3', 'rerank_score': 1.4069867134094238, 'rank': 2}
{'scheme_id': 'msYJf5gB0AGc0F0ivjhP', 'rerank_score': 0.7724201679229736, 'rank': 3}
{'scheme_id': 'GMZscpoB0AGc0F0iyj3X', 'rerank_score': -4.055066108703613, 'rank': 4}
{'scheme_id': 'RhhdDZwB5uF80_MG3Fwl', 'rerank_score': -4.143100738525391, 'rank': 5}


### Hybrid RRF + Cross-Encoder Evaluation

Apply Cross-Encoder reranking to every query in the golden dataset.

Preserve the original relevant scheme IDs and store the reranked scheme IDs and scores for downstream Ragas evaluation.

### Cross-Encoder Performance Benchmark

Measure Cross-Encoder inference time on 25 queries using the top 30 Hybrid RRF candidates.

This benchmark will help estimate the full evaluation runtime before processing all queries.

In [32]:
import time

benchmark_df = hybrid_df.head(25)

benchmark_pairs = []

for _, row in benchmark_df.iterrows():
    query = question_lookup[row["test_id"]]

    # Limit to top 30 RRF candidates for benchmarking
    candidate_ids = row["ranked_scheme_ids"][:30]

    for scheme_id in candidate_ids:
        profile = scheme_profiles.get(scheme_id)

        if profile:
            benchmark_pairs.append((query, profile))

print("Total pairs:", len(benchmark_pairs))

start_time = time.perf_counter()

benchmark_scores = cross_encoder.predict(
    benchmark_pairs,
    batch_size=32,
    show_progress_bar=False
)

benchmark_time = time.perf_counter() - start_time

print(f"Total time: {benchmark_time:.2f} seconds")
print(f"Pairs per second: {len(benchmark_pairs) / benchmark_time:.2f}")
print(f"Estimated full dataset time: {benchmark_time * 605 / 25 / 60:.2f} minutes")

Total pairs: 748
Total time: 2198.43 seconds
Pairs per second: 0.34
Estimated full dataset time: 886.70 minutes


### Cross-Encoder Performance Diagnostics

Inspect the Cross-Encoder device, PyTorch thread configuration, model input limit, and representative profile token lengths.

Identify whether CPU configuration or oversized inputs are contributing to slow inference.

In [33]:
import torch
import os
import numpy as np

print("Device:", cross_encoder.model.device)
print("CUDA available:", torch.cuda.is_available())
print("PyTorch CPU threads:", torch.get_num_threads())
print("PyTorch interop threads:", torch.get_num_interop_threads())
print("Logical CPU cores:", os.cpu_count())
print("Cross-Encoder max length:", cross_encoder.max_length)

# Inspect representative profile lengths
profile_lengths = [
    len(text)
    for text in scheme_profiles.values()
]

print("\nProfile character statistics:")
print("Mean:", round(np.mean(profile_lengths)))
print("Median:", round(np.median(profile_lengths)))
print("95th percentile:", round(np.percentile(profile_lengths, 95)))
print("Maximum:", max(profile_lengths))

# Inspect actual token count for sample query/profile pairs
sample_pairs = benchmark_pairs[:10]

token_counts = []

for query, profile in sample_pairs:
    encoded = cross_encoder.tokenizer(
        query,
        profile,
        truncation=True,
        max_length=cross_encoder.max_length
    )
    token_counts.append(len(encoded["input_ids"]))

print("\nToken counts after truncation:")
print(token_counts)
print("Average:", round(np.mean(token_counts), 2))

Device: cpu
CUDA available: False
PyTorch CPU threads: 4
PyTorch interop threads: 4
Logical CPU cores: 8
Cross-Encoder max length: 512


C:\Users\devendra\AppData\Local\Temp\ipykernel_12680\2231344533.py:10: DeprecationWarning: The `max_length` property was renamed and is now deprecated. Please use `max_seq_length` instead.
  print("Cross-Encoder max length:", cross_encoder.max_length)



Profile character statistics:
Mean: 1793
Median: 1533
95th percentile: 3500
Maximum: 3500

Token counts after truncation:
[398, 218, 268, 223, 512, 404, 178, 512, 383, 373]
Average: 346.9


C:\Users\devendra\AppData\Local\Temp\ipykernel_12680\2231344533.py:34: DeprecationWarning: The `max_length` property was renamed and is now deprecated. Please use `max_seq_length` instead.
  max_length=cross_encoder.max_length


In [ ]:
import time
import torch
import pandas as pd

test_pairs = benchmark_pairs[:8]

thread_results = []

for num_threads in [4]:
    torch.set_num_threads(num_threads)

    start = time.perf_counter()

    scores = cross_encoder.predict(
        test_pairs,
        batch_size=8,
        show_progress_bar=False
    )

    elapsed = time.perf_counter() - start

    thread_results.append({
        "threads": num_threads,
        "pairs": len(test_pairs),
        "time_seconds": round(elapsed, 2),
        "pairs_per_second": round(
            len(test_pairs) / elapsed, 3
        )
    })

thread_benchmark_df = pd.DataFrame(thread_results)

display(
    thread_benchmark_df.sort_values(
        "pairs_per_second",
        ascending=False
    )
)

,threads,pairs,time_seconds,pairs_per_second
2,4,8,29.07,0.275
3,8,8,35.61,0.225
1,2,8,36.09,0.222
0,1,8,63.82,0.125
